# NVL-H Data and Analysis Execution Control Notebook

This notebook is the centralized control plane for the NVL-H UPSVF + ILAS operating model.

## Usage
- Keep scripts deterministic and policy-driven.
- Use this notebook for planning, status, evidence, and decisions.
- Update logs daily after each run.

## End-Game Aim

1. Daily automated pull, merge, and publish is reliable.
2. Decisions are based on test-program trend windows, not single-day noise.
3. Retention is policy-controlled (before/after LIRA checkpoint).
4. Every output is traceable through request ID and manifest logs.

### Quantitative Success Criteria
- Daily pipeline reliability >= 99%.
- 100% runs produce manifest records.
- 0 silent data-loss events.
- All 6 analysis streams published on schedule.
- 100% retention-policy compliance.

## Architecture Summary

### Core Design Philosophy: Hybrid Model
AQUA is the system of record. Data is not duplicated wholesale into a second database.
Instead the model works in three layers:

- **Layer 1 — AQUA (source of truth):** All raw data lives here permanently. Every query can be re-run.
- **Layer 2 — Curated local artifacts:** Merged daily output files, program-window summaries, published analysis outputs. Kept according to retention policy.
- **Layer 3 — Metadata and history only:** For all post-LIRA programs after the retention window expires, only the manifest log and published analysis outputs are kept. Raw and merged files are deleted.

### Data Tiers
| Tier | Description | Example | Retention |
|---|---|---|---|
| A — Persistent | BinSplit/LIRA-critical programs | Pre-LIRA TPs | Keep everything permanently |
| B — Ephemeral | Post-LIRA programs in active window | Last 10 days | Raw + merged kept for 10 days then deleted |
| C — Metadata only | Post-LIRA programs after window expires | Older post-LIRA | Manifest + published outputs only |

### Why Not One Big Historical Database?
- Avoids maintaining a second full copy of all AQUA data
- Reduces storage, governance overhead, and naming-drift risk
- AQUA already stores everything permanently; re-pulling is always possible
- Trade-off: live analysis requires an AQUA query; not instant but fully reproducible

### Four-Step Linear Flow
```
[1. Daily Pull] --> [2. Curate + Merge] --> [3. Program Analytics] --> [4. Retain + Prune]
```

Each step has: required data, automation scripts/helpers, retention rule, and step-level success criteria.

### On-Demand vs Scheduled Mode
- Scheduled (daily): runs the full flow automatically
- On-demand: triggered manually for specific visual units, programs, or date ranges
- Both modes produce identical output format and manifest records
- Cache policy: if a merged output for the same program/date already exists and is within TTL, re-use it rather than re-pulling from AQUA


## Lifecycle Policy Framework

### Key Design Constraint: TP Naming Uncertainty
Critical test-program names may be estimated or partially known at policy creation time.
The policy must be flexible enough to work now and be tightened incrementally as names stabilize.

### Matching Priority Order (highest to lowest)
1. Manual override list — explicit exact TP names you know are critical now
2. Pattern match — wildcard or regex (e.g., `NVLHM66A*`, `*S622*`)
3. Program-family tag — if a family label is known (e.g., all "classhot" variants)
4. Date-range fallback — if a program ran between date X and date Y, treat it as critical
5. Default policy — if no rule matches, apply the safe default (non-critical, 10-day TTL)

### Operational Practices
- **Policy preview mode:** Before enforcing retention, run a dry-run to show which rule each file/program would hit. Review before deleting.
- **Monthly review:** Check for unmatched or new program names that need classification. Add them to the policy.
- **Effective dates:** Every rule should have `effective_from` and `effective_to` so you can model future changes without touching current behavior.
- **Version every policy change:** Increment version number. Every manifest record logs which policy version was used.

### Policy Config Fields
| Field | Meaning |
|---|---|
| rule_id | Unique rule identifier |
| match_type | exact / wildcard / regex / family / date_range / default |
| match_value | The value to match against (TP name, pattern, family tag, or date range) |
| criticality | critical (Tier A) / non_critical (Tier B/C) |
| retention_raw_days | Days to keep raw pulled files |
| retain_merged_days | Days to keep merged artifact |
| keep_forever | true = always keep (pre-LIRA TPs) |
| effective_from | Policy rule start date |
| effective_to | Policy rule end date (null = open-ended) |
| enabled | true/false toggle without deleting the rule |
| notes | Human explanation of why this rule exists |

### Current Policy Entries
> Fill in as TP names become known. Start with patterns, tighten to exact names later.

| rule_id | match_type | match_value | criticality | retention_raw_days | keep_forever | notes |
|---|---|---|---|---|---|---|
| R001 | default | * | non_critical | 10 | false | Safe fallback for unknown TPs |
| R002 | | | | | | <!-- Add LIRA-relevant TP pattern here --> |
| R003 | | | | | | <!-- Add post-LIRA TP pattern here --> |

### Policy Gaps — Action Items
- [ ] Confirm exact or estimated TP name for LIRA checkpoint program
- [ ] Define which program families are always critical
- [ ] Set effective dates for current production programs
- [ ] Schedule first monthly policy review date: ___________


## Request ID and Manifest Standard

### What is a Request ID?
A request ID is the unique tracking number for one pipeline run request.
Think of it as the "tracking number" for that run — every log entry, file, and decision produced by a run can be linked back to it.

- Format: UUID v4 — example: `f3f9c6d2-7b2f-4e1a-91f0-2f8f4c31a6ab`
- Generated once at run start, passed through all stages
- Also keep a human-readable run label alongside it (e.g., `2026-08-01_NVLH_daily_classhot`) for quick search

### What One Request ID Ties Together
1. Input scope (date / program / lot filters)
2. Pull operations (UPSVF + ILAS)
3. Merge output file
4. Analysis outputs for all active streams
5. Retention decision for all artifacts
6. Notifications and errors

### Retry Pattern
- New run = new request ID
- Retry of a failed run = new request ID + `parent_request_id` pointing to the failed run
- This keeps the audit trail clean and makes it easy to see which reruns were triggered by failures

### Minimum Manifest Columns
| Field | Meaning |
|---|---|
| request_id | Unique run identifier (UUID v4) |
| parent_request_id | Link for reruns/retries (null if not a retry) |
| run_label | Human-readable run name |
| trigger_type | scheduled / manual / rerun |
| started_at_utc | Run start time |
| finished_at_utc | Run end time |
| policy_version | Lifecycle policy version used this run |
| matched_rule_ids | Policy rules applied to each artifact |
| filters_json | Program / date range / lot filters used |
| raw_rows | Pulled raw row count (UPSVF + ILAS) |
| merged_rows | Final merged row count |
| unique_visual_ids | Unique unit count in merged output |
| ilas_coverage_rows | Rows with ILAS data populated |
| retention_decision | keep / delete / TTL-X-days per artifact |
| final_status | SUCCESS / FAILED / PARTIAL / WAITING_FOR_DATA |
| error_summary | Short failure explanation if any |

### What WAITING_FOR_DATA Means
If ILAS data is not yet available in AQUA for a given run, the pipeline continues with UPSVF-only output.
Status is WAITING_FOR_DATA, not FAILED. Run is still considered successful for UPSVF delivery.
ILAS merge can be completed later using the standalone merge utility once data appears.

### Manifest Storage
- Primary: append-only CSV or JSONL file alongside output directory
- Secondary (future): lightweight SQLite table for fast query
- Every run must write its manifest record even on failure

### Manifest Gaps — Action Items
- [ ] Define manifest file path and naming convention
- [ ] Confirm final_status values and what triggers each
- [ ] Confirm WAITING_FOR_DATA recovery procedure


## Script Skeleton Plan (Stage-by-Stage)

Each stage is a self-contained unit with standard input/output contracts.
All stages write structured status. The orchestrator reads status and decides whether to continue or stop.

---

### Stage A — Daily Pull
**Aim:** Pull UPSVF and ILAS raw data from AQUA for the current date and active programs.

**Data needed:**
- Date range (default: today)
- Program filters (driven by lifecycle policy)
- Lot filters (optional override)
- AQUA report paths: `hmarkovi\BSAT_UPS2_POR_PTL_NVL WW12` (UPSVF), `hmarkovi\ILAS_VMIN_DTS` (ILAS)

**Scripts/helpers:**
- UPS Parametric data pull script (`aqua_nvlh_weekly_pull.ps1`)
- ILAS data pull script (`aqua_nvlh_ilas_vmin_analysis.ps1`)
- AQUA command helper (local cached copy of AquaCmdLine.exe, unblocked via `Unblock-File`)
- Task scheduling helper (`register_aqua_nvlh_weekly_task.ps1`, `schedule-weekly-run.ps1`)
- Run health check + retry logic (mutex guard, timeout handling, retry on WAITING_FOR_DATA)

**Key constraints from past work:**
- Do NOT add `-lots` or `-lotsfromfs` to ILAS AQUA args — causes empty returns
- Do NOT add `-lastNDaysTestEnd` to ILAS queries — causes empty results
- ILAS filtering strategy: VisualID-only HashSet match (NOT VisualID+Lot key pair)
- AquaCmdLine.exe must be cached locally at `%LOCALAPPDATA%\NVLH\AquaCmdLine\` to avoid UNC security popups
- Mutex `Global\NVLH_Aqua_NVLH_Weekly_Pull` guards against overlapping runs

**Retention:** Raw files kept in recent-run storage. Retention tier determined by lifecycle policy.

**Success criteria:**
- UPSVF raw file present and non-empty
- ILAS raw file present OR WAITING_FOR_DATA status recorded (non-fatal)
- Run status written to health log

---

### Stage B — Curate and Merge
**Aim:** Validate raw data, clean it, normalize column names, and merge UPSVF + ILAS into one unified artifact.

**Data needed:**
- UPSVF raw file (from Stage A)
- ILAS summary CSV (from ILAS analysis script)
- Schema/naming normalization map
- Program selection logic (most-abundant program by row count)

**Scripts/helpers:**
- ILAS data pull + analysis script (`aqua_nvlh_ilas_vmin_analysis.ps1`)
  - Strict Classhot filter: `Filter-RowsByOpergroup` enforces `6248_CLASSHOT`
  - VisualID chunking via `MaxVisualIdsPerQuery`
- Data merge script (`merge-ilas-into-upsvf.ps1`)
  - Merge key: `VISUAL_ID||LOTFROMFS` (case-insensitive, whitespace-trimmed)
  - All UPSVF rows preserved even if no ILAS match (empty ILAS columns, not dropped)
- Schema guard + naming map (column presence checks, alias resolution)

**Key constraints from past work:**
- ILAS summary output uses `VisualID` and `LotFromFs` (mixed case) — normalize before matching
- UPSVF uses `VISUAL_ID` and `LOTFROMFS` (uppercase)
- Merge success requires > 0 rows with ILAS data; if 0 matches, check ILAS data availability before declaring failure
- Most-abundant program selection: group by PROGRAM column, sort by count DESC, pick first

**Retention:** Merged daily artifact kept per policy. Clean intermediate CSV kept only if `KeepCleanCsvArtifact` flag is set.

**Success criteria:**
- Merged output file created with > 0 rows
- No silent row loss (merged row count >= UPSVF input row count)
- Schema checks pass
- ILAS coverage metrics written to manifest

---

### Stage C — Program Analytics
**Aim:** Compute test-program-level metrics and publish all six analysis streams.

**Data needed:**
- Daily merged artifact (from Stage B)
- Historical merged artifacts for rolling window (7-day and 10-day)
- Program partition logic (group by PROGRAM column)
- Correlation and shift thresholds (configurable)

**Scripts/helpers:**
- Vmin analysis script (`build-vmin-all-products-validation.ps1`)
- Parametric summary script (`build-vmin-starting-point.ps1`)
- Correlation analysis script + Python validation (`build_vmin_all_products_validation.py`)
- Shift analysis job (lot-shift + program-shift detection)
- Email report generator (daily digest publisher)

**Why program windows, not single-day views:**
A single day can be biased by lot composition or temporary process shifts.
A 7-day or 10-day rolling window gives stable, decision-grade signal.
Minimum for publishing a trend: at least 3 distinct days and 2 lots.

**Six analysis streams published from this stage:**
1. TP release approval
2. Parametric results week-to-week follow-up
3. PreSi to PostSi matching
4. Parametric areas where improvement is required (BinSplit impact ranked)
5. Test time reduction relevant analysis
6. Thermal compliance

**Retention:** Published program analysis outputs kept long-term. Raw intermediate tables kept per policy.

**Success criteria:**
- All six streams produce output for every active program
- Program-window tables generated
- Daily summary email sent
- Shift alerts triggered if thresholds exceeded

---

### Stage D — Retention and Cleanup
**Aim:** Enforce lifecycle policy on all artifacts produced in this and previous runs.

**Data needed:**
- Lifecycle policy file (versioned JSON/YAML)
- Run manifest (to identify files and their policy-matched tier)
- List of all artifacts with creation dates

**Scripts/helpers:**
- Retention policy checker (applies matching priority, classifies each file)
- Manifest and history writer (appends final retention decision to manifest)
- Index generator (`generate-weekly-index.ps1` extended into full manifest index)
- Cleanup job for old files (deletes only files explicitly marked for deletion in manifest — never wildcard-deletes)

**Critical safety rule:** Only delete files that are explicitly listed in the manifest with a DELETE decision. No wildcard cleanup of entire folders.

**Retention rules:**
- Tier A (pre-LIRA or critical): keep raw + clean + merged + analysis forever
- Tier B (post-LIRA, within window): keep raw + merged for 10 days
- Tier C (post-LIRA, expired): delete raw and merged, keep metadata + published analysis only

**Success criteria:**
- Policy enforced for 100% of artifacts
- Audit log records what was deleted and why
- No policy violations
- Storage stays within target

---

### Stage E — Publish and Notify
**Aim:** Deliver daily analysis summary to stakeholders.

**Data needed:**
- Analysis stream outputs from Stage C
- Shift alert flags
- Distribution list and severity rules

**Scripts/helpers:**
- Email report generator

**Success criteria:**
- Daily digest delivered on schedule
- Shift alerts included where relevant
- Links to full analysis outputs included

---

### Stage F — Manifest Finalize
**Aim:** Close the manifest record for this run with final status and all metrics.

**Data needed:**
- Stage statuses from A–E
- Row counts, coverage metrics, retention decisions

**Scripts/helpers:**
- Manifest writer (append row to manifest CSV/JSONL)

**Golden rule:** Manifest is always written, even on failure. If the run crashed, the manifest record explains why.

**Success criteria:**
- 1 run = 1 manifest record, always
- Final status is one of: SUCCESS / FAILED / PARTIAL / WAITING_FOR_DATA
- All required fields populated

---

### Stage Gaps — Action Items
- [ ] Define standard input/output JSON contract per stage
- [ ] Confirm AquaCmdLine.exe cache path for all environments
- [ ] Confirm email distribution list and format
- [ ] Define shift alert thresholds per analysis stream


## Agent Strategy

### Core Governance Rule
**Scripts + versioned policy enforce actions. Agents recommend, not enforce.**
No agent should delete files, change policy, or trigger production actions autonomously.

### Recommended Agent Roles

#### 1. Policy Suggestion Agent
- Reviews new or unmatched TP names detected in recent runs
- Suggests which policy rule (or new rule) should apply
- Outputs a proposal document only — a human approves before adding to policy
- Input: list of unmatched programs from manifest + current policy rules
- Useful when TP naming is uncertain or programs shift families

#### 2. Data Quality Triage Agent
- Reviews anomalies flagged in daily analysis
- Explains likely root cause (lot bias, schema drift, ILAS data gap, etc.)
- Proposes additional checks or validation logic to add to scripts
- Input: manifest anomaly flags, row count deltas, ILAS coverage drops
- Does NOT modify scripts — outputs explanation + recommendations only

#### 3. Reporting / Narrative Agent
- Converts daily KPI metrics into stakeholder-readable summary
- Produces narrative for the email digest
- Input: analysis stream outputs, shift alerts, trend tables
- Output: plain-language summary paragraph per analysis stream

#### 4. Runbook / Failure Response Agent
- Given a FAILED or PARTIAL manifest record, suggests next operational action
- Uses known failure patterns from past runs (e.g., AquaCmdLine timeout, ILAS empty, schema mismatch)
- Input: manifest error_summary field + stage statuses
- Output: ordered troubleshooting steps

### Known Failure Patterns (Feed to Runbook Agent)
| Pattern | Likely Cause | Recovery Action |
|---|---|---|
| ILAS summary output was not generated | ILAS data not yet in AQUA | Set WAITING_FOR_DATA, re-run later |
| AquaCmdLine security popup | UNC-executed binary not cached locally | Run Resolve-AquaExePathForAutomation |
| 0 rows with ILAS data after merge | Program shift (e.g., S622 → S623) | Check ILAS availability for new program |
| Merged row count < UPSVF input | Silent row loss in clean step | Check filter logic in curate stage |
| Mutex already held | Overlapping run still active | Wait, then force-release if stale |

### Agent Prompt Placeholders
- Policy agent prompt: ___________
- Data quality agent prompt: ___________
- Narrative agent prompt: ___________
- Runbook agent prompt: ___________


## Curated Metadata — What Is Kept and Why

When raw and merged data files are deleted after the retention window, these artifacts must be kept permanently:

### Mandatory Long-Term Artifacts (Never Delete)
| Artifact | Description | Why Keep |
|---|---|---|
| Run manifest record | One row per run in manifest CSV/JSONL | Full audit trail, reproducibility |
| Published analysis outputs | PPT, CSV summaries, KPI tables | Evidence for decisions |
| Retention decision log | What was deleted, when, which rule applied | Governance and compliance |
| Policy version history | Every version of the lifecycle policy file | Explains why rules were applied |
| Decision trace table | What decision was made based on which output | Links analysis to actions |

### Metadata Per Run (Captured in Manifest)
| Field | Description |
|---|---|
| request_id | Unique identifier for the run |
| run_label | Human-readable name |
| trigger_type | How the run started |
| input_filters | Program, date, lot filters used |
| policy_version | Which policy version was applied |
| matched_rule_ids | Which rules classified each artifact |
| raw_rows | Size of pulled data |
| merged_rows | Size after merge |
| unique_visual_ids | Unique unit count |
| ilas_coverage_rows | Rows with ILAS data |
| retention_decision | Per-artifact keep/delete/TTL decision |
| final_status | Run outcome |
| error_summary | Failure reason if any |

### What Can Be Deleted After Window
- Raw UPSVF files
- Raw ILAS files
- Intermediate clean CSV files
- Merged daily artifact (after analysis is published and window expires)

### What Cannot Be Deleted
- Published analysis outputs (reports, summary CSVs)
- Manifest records
- Policy files and version history
- Retention decision audit log

### Data Dictionary (Fill in as Columns Are Used)
| Column Name | Source | Meaning | Transformation | Owner |
|---|---|---|---|---|
| VISUAL_ID | UPSVF AQUA | Unique visual unit identifier | Uppercase trim | Data Eng |
| LOTFROMFS | UPSVF AQUA | Lot number from fab system | Uppercase trim | Data Eng |
| PROGRAM | UPSVF AQUA | Test program name | Most-abundant selection logic | Data Eng |
| ILAS_* | ILAS AQUA | Parametric test signals | Prefixed after merge | Analytics |
| DevRevStep | UPSVF AQUA | Device revision and step identifier | Used for filtering | Analytics |
| | | | | |
| | | | | |


In [ ]:
# Request ID + manifest stub generator (fill paths and fields before use)
import uuid
from datetime import datetime, timezone

request_id = str(uuid.uuid4())
manifest_row = {
    "request_id": request_id,
    "parent_request_id": None,
    "run_label": "",
    "trigger_type": "scheduled",
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "finished_at_utc": "",
    "policy_version": "",
    "matched_rule_ids": [],
    "filters_json": {},
    "raw_rows": None,
    "merged_rows": None,
    "unique_visual_ids": None,
    "ilas_coverage_rows": None,
    "retention_decision": "",
    "final_status": "",
    "error_summary": ""
}

manifest_row

In [ ]:
# NVL-H pipeline orchestrator skeleton
# Replace each TODO with real implementation as stages are completed.
# Each stage writes $stageStatus = "SUCCESS" / "FAILED" / "WAITING_FOR_DATA" / "PARTIAL"
# Orchestrator reads status and decides whether to continue.

param(
  [string]$RunLabel      = "",       # e.g. "2026-08-01_NVLH_daily_classhot"
  [string]$PolicyVersion = "",       # e.g. "v1.0"
  [string]$ProgramFilter = "",       # e.g. "NVLHM66A*" or exact name
  [string]$DateRange     = "",       # e.g. "2026-07-25:2026-08-01"
  [string]$OutputDir     = "R:\Products\NVL\NVL-H\Weekly Runs",
  [switch]$DryRun                    # Preview mode: report what would run, do not execute
)

# ── Orchestrator startup ──────────────────────────────────────────────────────
$requestId = [guid]::NewGuid().Guid
$parentRequestId = $null             # Set if this is a retry of a previous run
$startTime = (Get-Date).ToUniversalTime().ToString("o")
Write-Host "Request ID : $requestId"
Write-Host "Run label  : $RunLabel"
Write-Host "Policy     : $PolicyVersion"
Write-Host "Dry run    : $DryRun"

$manifest = [ordered]@{
    request_id        = $requestId
    parent_request_id = $parentRequestId
    run_label         = $RunLabel
    trigger_type      = "manual"     # change to "scheduled" in task registration
    started_at_utc    = $startTime
    finished_at_utc   = $null
    policy_version    = $PolicyVersion
    filters_json      = @{ program = $ProgramFilter; date_range = $DateRange }
    final_status      = "RUNNING"
    error_summary     = ""
}

# ── Stage A: Daily Pull ───────────────────────────────────────────────────────
# Aim: Pull UPSVF + ILAS raw data from AQUA for the current date and active programs.
# Scripts: aqua_nvlh_weekly_pull.ps1 + aqua_nvlh_ilas_vmin_analysis.ps1
# Constraints: no -lots/-lotsfromfs in ILAS args; AquaCmdLine.exe must be locally cached.
$stageA_status = "TODO"
# TODO: call UPSVF pull script
# TODO: call ILAS pull script
# TODO: set $stageA_status = "SUCCESS" / "WAITING_FOR_DATA" / "FAILED"
Write-Host "Stage A (Pull): $stageA_status"

if ($stageA_status -eq "FAILED") {
    $manifest.final_status = "FAILED"; $manifest.error_summary = "Stage A pull failed"
    # TODO: write manifest and exit
    return
}

# ── Stage B: Curate and Merge ─────────────────────────────────────────────────
# Aim: Clean, normalize, and merge UPSVF + ILAS into one unified artifact.
# Scripts: aqua_nvlh_ilas_vmin_analysis.ps1 + merge-ilas-into-upsvf.ps1
# Key: merge key is VISUAL_ID||LOTFROMFS (case-insensitive); all UPSVF rows must be preserved.
$stageB_status = "TODO"
# TODO: call ILAS analysis script
# TODO: call merge script
# TODO: validate merged row count >= UPSVF input row count
# TODO: set $stageB_status
Write-Host "Stage B (Merge): $stageB_status"

# ── Stage C: Program Analytics ────────────────────────────────────────────────
# Aim: Compute program-window metrics and publish all 6 analysis streams.
# Scripts: build-vmin-all-products-validation.ps1, correlation jobs, shift jobs
# Key: 7-day and 10-day rolling windows; minimum 3 days + 2 lots before publishing trend.
$stageC_status = "TODO"
# TODO: call Vmin analysis script
# TODO: call correlation + shift jobs
# TODO: set $stageC_status
Write-Host "Stage C (Analytics): $stageC_status"

# ── Stage D: Retain + Prune ───────────────────────────────────────────────────
# Aim: Apply lifecycle policy to all artifacts from this and previous runs.
# Scripts: retention policy checker, manifest reader, cleanup job
# Key: NEVER delete without manifest evidence. Only delete files listed with DELETE decision.
$stageD_status = "TODO"
# TODO: read lifecycle policy (versioned JSON/YAML)
# TODO: classify each artifact by tier (A = keep, B = 10-day, C = metadata only)
# TODO: delete only what policy marks for deletion
# TODO: write retention audit log
# TODO: set $stageD_status
Write-Host "Stage D (Retention): $stageD_status"

# ── Stage E: Publish and Notify ───────────────────────────────────────────────
# Aim: Send daily digest to stakeholders.
$stageE_status = "TODO"
# TODO: call email publisher
# TODO: include shift alerts if any thresholds exceeded
# TODO: set $stageE_status
Write-Host "Stage E (Publish): $stageE_status"

# ── Stage F: Manifest Finalize ────────────────────────────────────────────────
# Aim: Close the manifest record. Always runs, even on failure.
$manifest.finished_at_utc = (Get-Date).ToUniversalTime().ToString("o")
$manifest.final_status    = "SUCCESS"   # TODO: derive from stage statuses
# TODO: append manifest row to manifest CSV/JSONL
Write-Host "Manifest written: $requestId"


In [ ]:
-- SQL templates for manifest analytics
-- 1) Daily success rate
SELECT
  date(started_at_utc) AS run_day,
  COUNT(*) AS total_runs,
  SUM(CASE WHEN final_status = 'SUCCESS' THEN 1 ELSE 0 END) AS success_runs
FROM run_manifest
GROUP BY 1
ORDER BY 1 DESC;

-- 2) Recent failures
SELECT request_id, started_at_utc, final_status, error_summary
FROM run_manifest
WHERE final_status IN ('FAILED', 'PARTIAL')
ORDER BY started_at_utc DESC
LIMIT 50;

## Analysis Streams (Scope Locked)

All six streams are produced from the unified UPSVF + ILAS pipeline daily.
They are treated as mandatory release gates in the test-program dashboard.

---

### Stream 1: TP Release Approval
**What it is:** Daily assessment of whether the current test program is ready for approval or release.

**How it is produced:** Compare current program's Vmin distribution, ILAS coverage, and parametric results against defined release thresholds.

**Key inputs:** Merged daily artifact, BinSplit model outputs, Vmin targets per domain and frequency.

**Success gate:** Pass/fail verdict per program, visible every day, not just at manual review.

**KPI:** _________________

---

### Stream 2: Parametric Results Week-to-Week Follow-Up
**What it is:** Tracks how parametric metrics (Vmin, UPM, deviation) change from one week to the next.

**How it is produced:** Compare program-window rolling tables across weeks. Flag significant shifts.

**Key inputs:** 7-day program-window KPI tables from current and prior weeks.

**Why program window, not single day:** A single day can be lot-biased. Window view is stable and comparable.

**KPI:** _________________

---

### Stream 3: PreSi to PostSi Matching
**What it is:** Validates that silicon test results match pre-silicon (simulation/model) predictions.

**How it is produced:** Join parametric measurements from PostSi runs against PreSi model predictions. Compute delta and correlation.

**Key inputs:** Merged UPSVF+ILAS dataset, PreSi prediction table.

**KPI:** _________________

---

### Stream 4: Parametric Improvement Areas (BinSplit Impact Ranked)
**What it is:** Identifies which parametric areas have the most room for improvement, ranked by their impact on the BinSplit model.

**How it is produced:** Score each parametric domain/test by its BinSplit impact coefficient. Sort descending. Flag top improvement candidates.

**Key inputs:** BinSplit model (LIRA reference), correlation analysis outputs, Vmin deviations per domain.

**Why BinSplit rank matters:** Not all parametric deviations are equally important. This stream focuses effort on what actually moves yield and bin counts.

**KPI:** _________________

---

### Stream 5: Test Time Reduction Analysis
**What it is:** Identifies tests or test sequences where time can be reduced without yield impact.

**How it is produced:** Analyze test-time distributions, correlate with parametric outcomes, flag redundant or low-information tests.

**Key inputs:** ILAS detail records, test execution times, pass/fail rates per test.

**KPI:** _________________

---

### Stream 6: Thermal Compliance
**What it is:** Verifies that thermal behavior of the device is within spec at all test conditions.

**How it is produced:** Extract thermal-relevant parametric columns, compare against compliance thresholds.

**Key inputs:** Merged dataset, thermal compliance spec table.

**KPI:** _________________

---

### Stream Gaps — Action Items
- [ ] Define numeric KPI threshold for each stream
- [ ] Confirm PreSi prediction table format and source
- [ ] Confirm BinSplit model reference file location and version
- [ ] Define shift alert thresholds for Streams 1, 2, 4


## Detailed Next Steps and Ownership

### Why These Steps, in This Order
Row 1 defines the rules. Rows 2–7 implement and enforce those rules. Row 8 signs off for production.
Steps 1–3 (Week 1) are the foundation — nothing downstream works without them.

| Step | Aim | What It Means | Owner | Target | Status | Notes |
|---|---|---|---|---|---|---|
| 1 | Define test-program lifecycle policy | Create the rules for how each program is treated — which are critical (keep everything), which are non-critical (10-day TTL). Use flexible matching because exact TP names may not be final yet. Output is a versioned policy config file. | Data + Product | Week 1 | TODO | Start with patterns, tighten to exact names later |
| 2 | Harden daily pull orchestration | Make UPSVF + ILAS pull reliable and repeatable. Add retry logic, health checks, timeout handling, and status logging. Without this, all downstream stages are unstable. | Automation | Week 1 | TODO | Existing scripts need mutex guard and retry hardening |
| 3 | Manifest and history logging | Write one manifest record for every run. Include request ID, input filters, row counts, ILAS coverage, retention decision, and final status. This is the audit trail for everything. | Data Eng | Week 1 | TODO | Schema must be agreed before coding |
| 4 | Program-window analytics | Move from day-by-day noise to program-level trend intelligence. Compute 7-day and 10-day rolling windows per program. Minimum for publishing: 3 distinct days and 2 lots. | Analytics | Week 2 | TODO | Depends on Step 2 stability |
| 5 | Correlation and shift jobs | Run automated jobs that detect lot-to-lot and program-to-program parametric shifts. This is the early warning layer for release risk, data drift, or process instability. | Analytics | Week 2 | TODO | Alert thresholds need to be defined |
| 6 | Email publisher | Distribute the daily analysis digest automatically. Include KPIs, shift alerts, release-relevant conclusions, and links to full outputs. | Automation | Week 2 | TODO | Distribution list and severity rules needed |
| 7 | Retention enforcer | Scheduled job that checks each artifact against policy and deletes what should expire. Writes what was deleted and why to the audit log. Never deletes without manifest evidence. | Data Eng | Week 3 | TODO | Only runs after manifest logging is stable |
| 8 | Governance review | Formal review by leadership and technical owners. Covers reliability, quality of six analysis streams, retention compliance, and decision usefulness. Converts a good implementation into an approved production process. | Leadership | Week 4 | TODO | Define review criteria before Week 4 |


## Daily Run Journal

| Date | Request ID | Pull | Merge | Analytics | Retention | Final Status | Key Issue |
|---|---|---|---|---|---|---|---|
|  |  |  |  |  |  |  |  |
|  |  |  |  |  |  |  |  |
|  |  |  |  |  |  |  |  |

### Placeholder: Daily follow-up actions
- [ ] 
- [ ] 

## Policy Change Log

| Date | Policy Version | Change Summary | Reason | Approved By |
|---|---|---|---|---|
|  |  |  |  |  |
|  |  |  |  |  |

## Decision Trace Log

| Date | Request ID | Key Finding | Decision | Owner |
|---|---|---|---|---|
|  |  |  |  |  |
|  |  |  |  |  |

## Risk and Gap Register

| Risk/GAP | Impact | Detection | Mitigation | Owner | Due Date |
|---|---|---|---|---|---|
|  |  |  |  |  |  |
|  |  |  |  |  |  |
|  |  |  |  |  |  |

## Next Session Starter

Before ending each work session, update:
1. Daily Run Journal
2. Open Gaps
3. Next 3 actions

### Next 3 actions
1. 
2. 
3. 